In [1]:

import numpy as np 
import pandas as pd 
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
import kagglehub

/kaggle/input/datasets/harshinirajuladevi/movie125159/train_data.txt
/kaggle/input/datasets/hijest/genre-classification-dataset-imdb/Genre Classification Dataset/description.txt
/kaggle/input/datasets/hijest/genre-classification-dataset-imdb/Genre Classification Dataset/test_data_solution.txt
/kaggle/input/datasets/hijest/genre-classification-dataset-imdb/Genre Classification Dataset/test_data.txt
/kaggle/input/datasets/hijest/genre-classification-dataset-imdb/Genre Classification Dataset/train_data.txt


## CodSoft Machine Learning Internship – Task 1

### 1.Import Required Libraries

In [2]:
import pandas as pd
import numpy as np
import re
import string
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

### List of files

In [3]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    print(dirname)
    for filename in filenames:
        print("   ", filename)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/harshinirajuladevi
/kaggle/input/datasets/harshinirajuladevi/movie125159
    train_data.txt
/kaggle/input/datasets/hijest
/kaggle/input/datasets/hijest/genre-classification-dataset-imdb
/kaggle/input/datasets/hijest/genre-classification-dataset-imdb/Genre Classification Dataset
    description.txt
    test_data_solution.txt
    test_data.txt
    train_data.txt


### 2.Load the dataset

In [4]:
import pandas as pd

train_df = pd.read_csv(
    "/kaggle/input/datasets/hijest/genre-classification-dataset-imdb/Genre Classification Dataset/train_data.txt",
    sep=" ::: ",
    engine="python",
    names=["ID", "TITLE", "GENRE", "DESCRIPTION"]
)

### 3.Explore the dataset

In [5]:
train_df.head()

,ID,TITLE,GENRE,DESCRIPTION
0,1,Oscar et la dame rose (2009),drama,Listening in to a conversation between his doc...
1,2,Cupid (1997),thriller,A brother and sister with a past incestuous re...
2,3,"Young, Wild and Wonderful (1980)",adult,As the bus empties the students for their fiel...
3,4,The Secret Sin (1915),drama,To help their unemployed father make ends meet...
4,5,The Unrecovered (2007),drama,The film's title refers not only to the un-rec...


In [6]:
train_df.tail()

,ID,TITLE,GENRE,DESCRIPTION
54209,54210,"""Bonino"" (1953)",comedy,This short-lived NBC live sitcom centered on B...
54210,54211,Dead Girls Don't Cry (????),horror,The NEXT Generation of EXPLOITATION. The siste...
54211,54212,Ronald Goedemondt: Ze bestaan echt (2008),documentary,"Ze bestaan echt, is a stand-up comedy about gr..."
54212,54213,Make Your Own Bed (1944),comedy,Walter and Vivian live in the country and have...
54213,54214,Nature's Fury: Storm of the Century (2006),history,"On Labor Day Weekend, 1935, the most intense h..."


In [7]:
train_df.shape

(54214, 4)

In [8]:
train_df.columns

Index(['ID', 'TITLE', 'GENRE', 'DESCRIPTION'], dtype='object')

In [9]:
train_df.info

<bound method DataFrame.info of           ID                                       TITLE        GENRE  \
0          1                Oscar et la dame rose (2009)        drama   
1          2                                Cupid (1997)     thriller   
2          3            Young, Wild and Wonderful (1980)        adult   
3          4                       The Secret Sin (1915)        drama   
4          5                      The Unrecovered (2007)        drama   
...      ...                                         ...          ...   
54209  54210                             "Bonino" (1953)       comedy   
54210  54211                 Dead Girls Don't Cry (????)       horror   
54211  54212   Ronald Goedemondt: Ze bestaan echt (2008)  documentary   
54212  54213                    Make Your Own Bed (1944)       comedy   
54213  54214  Nature's Fury: Storm of the Century (2006)      history   

                                             DESCRIPTION  
0      Listening in to a convers

#### Missing values

In [10]:
train_df.isnull().sum()

ID             0
TITLE          0
GENRE          0
DESCRIPTION    0
dtype: int64

#### duplicate values

In [11]:
train_df.duplicated().sum()

np.int64(0)

In [12]:
train_df["GENRE"].nunique()

27

#### Genre Distribution

In [13]:
train_df["GENRE"].value_counts()

GENRE
drama          13613
documentary    13096
comedy          7447
short           5073
horror          2204
thriller        1591
action          1315
western         1032
reality-tv       884
family           784
adventure        775
music            731
romance          672
sci-fi           647
adult            590
crime            505
animation        498
sport            432
talk-show        391
fantasy          323
mystery          319
musical          277
biography        265
history          243
game-show        194
news             181
war              132
Name: count, dtype: int64

In [15]:
train_df["DESCRIPTION"].str.len().describe()

count    54214.000000
mean       599.452429
std        446.026620
min         41.000000
25%        324.000000
50%        463.000000
75%        712.000000
max      10503.000000
Name: DESCRIPTION, dtype: float64

### 5.Data Preprocessing

In [17]:
train_df["DESCRIPTION"] = train_df["DESCRIPTION"].str.lower()

In [18]:
train_df["DESCRIPTION"] = train_df["DESCRIPTION"].apply(
    lambda x: re.sub(r'[^\w\s]', '', x)
)

In [19]:
train_df["DESCRIPTION"] = train_df["DESCRIPTION"].apply(
    lambda x: re.sub(r'\d+', '', x)
)

In [20]:
train_df["DESCRIPTION"] = train_df["DESCRIPTION"].apply(
    lambda x: re.sub(r'\s+', ' ', x).strip()
)
train_df["DESCRIPTION"]

0        listening in to a conversation between his doc...
1        a brother and sister with a past incestuous re...
2        as the bus empties the students for their fiel...
3        to help their unemployed father make ends meet...
4        the films title refers not only to the unrecov...
                               ...                        
54209    this shortlived nbc live sitcom centered on bo...
54210    the next generation of exploitation the sister...
54211    ze bestaan echt is a standup comedy about grow...
54212    walter and vivian live in the country and have...
54213    on labor day weekend the most intense hurrican...
Name: DESCRIPTION, Length: 54214, dtype: object

### 6.Split the Dataset

In [ ]:
import nltk
nltk.download('stopwords')

In [ ]:
stop_words = set(stopwords.words('english'))

In [23]:
train_df["DESCRIPTION"] = train_df["DESCRIPTION"].apply(
    lambda x: " ".join(
        word for word in x.split()
        if word not in stop_words
    )
)
train_df["DESCRIPTION"]

0        listening conversation doctor parents yearold ...
1        brother sister past incestuous relationship cu...
2        bus empties students field trip museum natural...
3        help unemployed father make ends meet edith tw...
4        films title refers unrecovered bodies ground z...
                               ...                        
54209    shortlived nbc live sitcom centered bonino wor...
54210    next generation exploitation sisters kapa bay ...
54211    ze bestaan echt standup comedy growing facing ...
54212    walter vivian live country difficult time keep...
54213    labor day weekend intense hurricane ever make ...
Name: DESCRIPTION, Length: 54214, dtype: object

In [24]:
X = train_df["DESCRIPTION"]
y = train_df["GENRE"]

In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

### 7. Convert Text into Numerical Features(TF-IDF)

In [26]:
vectorizer = TfidfVectorizer(stop_words="english")

In [27]:
X_train_tfidf = vectorizer.fit_transform(X_train)

In [28]:
X_test_tfidf = vectorizer.transform(X_test)

In [29]:
print(X_train_tfidf.shape)

(43371, 128743)


### 8.Train the Machine Learning Model 
Train Logistic Regression Model

In [30]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)

In [31]:
lr_model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000, random_state=42)

### 9.Model Evaluation

In [41]:
lr_predictions = lr_model.predict(X_test_tfidf)

In [33]:
print(classification_report(y_test, lr_predictions, zero_division=0))

              precision    recall  f1-score   support

      action       0.59      0.22      0.32       263
       adult       0.79      0.17      0.28       112
   adventure       0.45      0.10      0.16       139
   animation       1.00      0.01      0.02       104
   biography       0.00      0.00      0.00        61
      comedy       0.51      0.57      0.54      1443
       crime       0.50      0.01      0.02       107
 documentary       0.64      0.86      0.74      2659
       drama       0.52      0.81      0.64      2697
      family       0.50      0.05      0.10       150
     fantasy       0.00      0.00      0.00        74
   game-show       0.92      0.30      0.45        40
     history       0.00      0.00      0.00        45
      horror       0.68      0.56      0.61       431
       music       0.69      0.42      0.53       144
     musical       0.00      0.00      0.00        50
     mystery       0.00      0.00      0.00        56
        news       0.00    

In [34]:

print(classification_report(y_test, lr_predictions, zero_division=0))

              precision    recall  f1-score   support

      action       0.59      0.22      0.32       263
       adult       0.79      0.17      0.28       112
   adventure       0.45      0.10      0.16       139
   animation       1.00      0.01      0.02       104
   biography       0.00      0.00      0.00        61
      comedy       0.51      0.57      0.54      1443
       crime       0.50      0.01      0.02       107
 documentary       0.64      0.86      0.74      2659
       drama       0.52      0.81      0.64      2697
      family       0.50      0.05      0.10       150
     fantasy       0.00      0.00      0.00        74
   game-show       0.92      0.30      0.45        40
     history       0.00      0.00      0.00        45
      horror       0.68      0.56      0.61       431
       music       0.69      0.42      0.53       144
     musical       0.00      0.00      0.00        50
     mystery       0.00      0.00      0.00        56
        news       0.00    

In [35]:
train_df["GENRE"].value_counts()

GENRE
drama          13613
documentary    13096
comedy          7447
short           5073
horror          2204
thriller        1591
action          1315
western         1032
reality-tv       884
family           784
adventure        775
music            731
romance          672
sci-fi           647
adult            590
crime            505
animation        498
sport            432
talk-show        391
fantasy          323
mystery          319
musical          277
biography        265
history          243
game-show        194
news             181
war              132
Name: count, dtype: int64

In [36]:

print("Logistic Regression Accuracy:", accuracy_score(y_test, lr_predictions))

Logistic Regression Accuracy: 0.574748685788066


In [43]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test,  lr_predictions)
print("Accuracy:", accuracy)

Accuracy: 0.574748685788066


### 10.Predict Genre of a New Movie

In [ ]:
new_movie = [
    "A detective investigates a mysterious murder in an abandoned mansion."
]

new_movie_vector = vectorizer.transform(new_movie)
print(lr_model.predict(new_movie_vector)[0])

In [ ]:
new_movie = [
    "Two friends travel across the country, making everyone laugh with their funny adventures."
]

new_movie_vector = vectorizer.transform(new_movie)
print(lr_model.predict(new_movie_vector)[0])

In [ ]:
import joblib

joblib.dump(lr_model, "logistic_regression_movie_genre_model.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

### 11.Load the Official Test Dataset

In [ ]:
test_df = pd.read_csv(
    "/kaggle/input/datasets/hijest/genre-classification-dataset-imdb/Genre Classification Dataset/test_data.txt",
    sep=" ::: ",
    engine="python",
    names=["ID", "TITLE", "DESCRIPTION"]
)

### 12.Preprocess Test Data

In [ ]:
import re

test_df["DESCRIPTION"] = test_df["DESCRIPTION"].str.lower()

test_df["DESCRIPTION"] = test_df["DESCRIPTION"].apply(
    lambda x: re.sub(r"[^\w\s]", "", x)
)

test_df["DESCRIPTION"] = test_df["DESCRIPTION"].apply(
    lambda x: re.sub(r"\d+", "", x)
)

test_df["DESCRIPTION"] = test_df["DESCRIPTION"].apply(
    lambda x: re.sub(r"\s+", " ", x).strip()
)
test_df["DESCRIPTION"]

### 14.Predict Genres for the Test Dataset

In [ ]:
X_test_final = vectorizer.transform(test_df["DESCRIPTION"])

In [ ]:
predicted_genres = lr_model.predict(X_test_final)

In [ ]:
test_df["PREDICTED_GENRE"] = predicted_genres

In [ ]:
test_df.head()

### 14.Evaluate Using the Official Test Solution

In [ ]:
solution_df = pd.read_csv(
    "/kaggle/input/datasets/hijest/genre-classification-dataset-imdb/Genre Classification Dataset/test_data_solution.txt",
    sep=" ::: ",
    engine="python",
    names=["ID", "TITLE", "GENRE", "DESCRIPTION"]
)

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(
    solution_df["GENRE"],
    predicted_genres
)

print("Final Test Accuracy:", accuracy)

### 15.Graph

In [ ]:
import matplotlib.pyplot as plt

genre_counts = train_df["GENRE"].value_counts()

plt.figure(figsize=(12,8))

genre_counts.plot(kind="bar")

plt.title("Distribution of Movie Genres")
plt.xlabel("Genre")
plt.ylabel("Number of Movies")

plt.xticks(rotation=90)

plt.show()

# Conclusion

In this project, a Machine Learning model was developed to classify movie genres based on movie plot summaries.

The dataset was preprocessed using Natural Language Processing (NLP) techniques, and the text was converted into numerical features using the TF-IDF Vectorizer. A Logistic Regression classifier was trained on the processed data and evaluated using accuracy and classification metrics.

The trained model successfully predicted genres for unseen movie descriptions and was further evaluated using the official test dataset provided with the project. This project demonstrates a complete supervised machine learning workflow, including data preprocessing, feature extraction, model training, evaluation, and prediction.